# Colony counter: training on Google Colab

Everything runs in Google's cloud; nothing runs on your computer.

1. In Google Drive, create a folder **colony-data** (in *My Drive*) and upload the labelled plate zips into it.
2. Here: *Runtime > Change runtime type > T4 GPU*.
3. *Runtime > Run all*, and allow access to Google Drive when asked.

Training takes about 5 hours. Checkpoints are saved to *My Drive/colony-counter* after every epoch: if Colab disconnects, open the notebook again and *Run all*; it continues where it stopped.
At the end the model (`colony.onnx`, `colony.json`) and the evaluation report are in *My Drive/colony-counter/model*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/colony-data'
OUT = '/content/drive/MyDrive/colony-counter'
import os, glob
os.makedirs(OUT, exist_ok=True)
zips = sorted(glob.glob(f'{DATA}/*.zip'))
assert zips, f'No zip files found in {DATA}'
print(zips)

In [ ]:
%cd /content
!test -d colony-counter || git clone -q https://github.com/mrbanoth/colony-counter.git
%cd /content/colony-counter
!git pull -q
!pip install -q -r training/requirements.txt
!pip uninstall -q -y onnxruntime && pip install -q onnxruntime-gpu

In [ ]:
# Copy the zips to the fast local disk, then build the train/val/test data (a few minutes).
!mkdir -p /content/zips && cp -n "{DATA}"/*.zip /content/zips/
!test -f data/yolo/data.yaml || python training/prepare_dataset.py /content/zips/*.zip --out data --workers 4

In [ ]:
# Starts training, or resumes it from the last checkpoint on Drive.
import subprocess
last = f'{OUT}/runs/colab/weights/last.pt'
if os.path.exists(last):
    cmd = ['python', 'training/train.py', '--resume', last]
else:
    cmd = ['python', 'training/train.py', '--preset', 'colab', '--project', f'{OUT}/runs']
result = subprocess.run(cmd)
print('training finished' if result.returncode == 0 else 'see the messages above (a finished run cannot be resumed; that is fine)')

In [ ]:
# Export to ONNX, tune the counting thresholds on the validation plates, evaluate on the test plates.
!python training/export.py "{OUT}/runs/colab/weights/best.pt" --out weights
!python training/tune.py --model weights --plates data/plates/val
!python training/evaluate.py --model weights --plates data/plates/test --out reports

In [ ]:
!mkdir -p "{OUT}/model" && cp weights/colony.onnx weights/colony.json "{OUT}/model/" && cp -r reports "{OUT}/model/"
!cp "{OUT}/runs/colab/weights/best.pt" "{OUT}/model/colony.pt" && cp "{OUT}/runs/colab/results.csv" "{OUT}/model/"
import json
m = json.load(open('reports/metrics.json'))
print(json.dumps({k: m[k] for k in ('detection', 'recall_by_size', 'count')}, indent=2))
print('Model and report saved to My Drive/colony-counter/model')